# IBM quantum

In [ ]:
import os
from qiskit import QuantumCircuit
from qiskit.transpiler.preset_passmanagers import generate_preset_pass_manager
from qiskit_ibm_runtime import QiskitRuntimeService, SamplerV2 as Sampler


service = QiskitRuntimeService(
    channel="ibm_quantum_platform",
    token="your token",
    instance="your instance",
)


# --- Bell circuit: |Φ+> = (|00> + |11>) / sqrt(2) ---
qc = QuantumCircuit(2)
qc.h(0)
qc.cx(0, 1)
qc.measure_all()

print(qc.draw())


# --- select a real IBM backend ---
backend = service.least_busy(
    operational=True,
    simulator=False,
    min_num_qubits=2,
)

print("Backend:", backend.name)


# --- transpile to an ISA circuit for the selected hardware ---
pm = generate_preset_pass_manager(
    backend=backend,
    optimization_level=1,
)
isa_qc = pm.run(qc)


# --- run on a real QPU ---
sampler = Sampler(mode=backend)

job = sampler.run([isa_qc], shots=1024)
print("Job ID:", job.job_id())

result = job.result()
counts = result[0].data.meas.get_counts()

print("Counts:", counts)

## Spinqit

In [ ]:
from spinqit.qiskit.circuit import QuantumCircuit
from spinqit import get_compiler, get_nmr, NMRConfig


qc = QuantumCircuit(2)
qc.h(0)
qc.cx(0, 1)

compiler = get_compiler("qiskit")
exe = compiler.compile(qc, 0)

engine = get_nmr()

config = NMRConfig()
config.configure_shots(1024)
config.configure_ip("192.168.137.1")
config.configure_port(55444)
config.configure_account("USERNAME", "PASSWORD")
config.configure_task("bell_qiskit_style", "Bell circuit via SpinQit qiskit syntax")

result = engine.execute(exe, config)

print("Counts:", result.counts)